# `fg.rules.structure` — 规则的静态结构(与 explain 对位)

`fg.rules.structure(rule)` 返回一个**引擎中立**的 `RuleStructure`:规则的静态结构(branches / occurrences / atoms / joins),**不运行任何引擎、不读任何事实**。

它与 `fg.eval.explain(...)` 的 `EvidenceGraph` **逐节点对位**(同一棵树):
- **display**(`RuleStructure`)= 裸逻辑:变量(`FreeVar`),无 verdict;
- **explanation**(explain 的 `EvidenceGraph`)= 执行后的值 + verdict。

二者按 `branch_id` / `occurrence_alias` / `atom_id` / `join_id` 对齐——因为都从同一个 lowering plan 派生(单一来源,不漂移)。

## 1. 构造一个复合规则

规则:**一个 manager 和一个 report 在同一部门**(分支 `c0`,join 在 `dept`),**或一个 executive**(分支 `c1`);head 把 `dept` 钉到 `"eng"`(闭合头)。

In [1]:
import factgraph.sdk as sdk
from factgraph.application import build_schema_index, entity_info, field_predicate, resolve_selector
from factgraph.application.protocol import EntitySelector, FreeVar, Rule
from factgraph.core.evidence.write_protocol import set_field
from factgraph.core.rules.where_ast import Const, CmpAtom, PredAtom, Var

class Person(sdk.Entity):
    person_id: str = sdk.Identity()
    role: str = sdk.Field()
    dept: str = sdk.Field()

fg = sdk.SDKStore([Person])
index = build_schema_index(fg.schema_ir)
info = entity_info(index, "Person")
role_pred = field_predicate(index, "Person", "role").pred_id
dept_pred = field_predicate(index, "Person", "dept").pred_id

def seed(pid, role, dept):
    ref = resolve_selector(EntitySelector(entity_type="Person", identity={"person_id": pid}), index=index)
    enc = ref.encoded_ref or ""
    set_field(fg.ledger, info.exists_predicate_id, enc, [])
    set_field(fg.ledger, info.identity_predicates["person_id"].pred_id, enc, [("string", pid)])
    set_field(fg.ledger, role_pred, enc, [("string", role)])
    set_field(fg.ledger, dept_pred, enc, [("string", dept)])

seed("alice", "manager", "eng"); seed("bob", "report", "eng")
seed("zoe", "executive", "eng"); seed("carol", "report", "sales")

# leaf rules (occurrence ports use distinct names so the composite has no port clash)
m, md_, mr = Var("$m"), Var("$md"), Var("$mr")
r, rd, rr = Var("$r"), Var("$rd"), Var("$rr")
x, xd, xr = Var("$x"), Var("$xd"), Var("$xr")
hd = Var("$hd")
mgr = Rule(id="manager",   when=(PredAtom(role_pred,[m,mr]), CmpAtom("eq",mr,Const("manager")),   PredAtom(dept_pred,[m,md_])), ports={"mgr":m,"dept":md_}, repr="manager %mgr in %dept")
rep = Rule(id="report",    when=(PredAtom(role_pred,[r,rr]), CmpAtom("eq",rr,Const("report")),    PredAtom(dept_pred,[r,rd])),  ports={"rep":r,"dept":rd},  repr="report %rep in %dept")
exe = Rule(id="executive", when=(PredAtom(role_pred,[x,xr]), CmpAtom("eq",xr,Const("executive")), PredAtom(dept_pred,[x,xd])), ports={"exec":x,"dept":xd}, repr="executive %exec in %dept")
head = Rule(id="staffed_dept", when=(CmpAtom("eq",hd,Const("eng")),), ports={"dept":hd})

# composite: (manager & report joined on dept) OR executive
expr = (mgr.as_("m") & rep.as_("r")).join(mgr.as_("m").dept.eq(rep.as_("r").dept)) | exe.as_("x")
print("composite built")

composite built


## 2. `fg.rules.structure(...)` —— 静态结构(零引擎)

In [2]:
structure = fg.rules.structure(expr, head=head)

print("source_kind     :", structure.source_kind)
print("head_rule_id    :", structure.head_rule_id, "(binding:", structure.head_binding_kind + ")")
print("head_closure    :", structure.head_closure)          # 闭合头 -> is_closed=True
print("render_compact()  :", structure.render_compact())       # authored AST (== inspect)
print("render()  (naked):", structure.render())

source_kind     : rule_expr
head_rule_id    : staffed_dept (binding: external)
head_closure    : HeadClosure(is_closed=True, unbound_ports=())
render_compact()  : ((m:manager & r:report).join(1) | x:executive)
render()  (naked): RuleExprInspect | ((m:manager & r:report).join(1) | x:executive) | m:manager manager <mgr> in <dept>; r:report report <rep> in <dept>; x:executive executive <exec> in <dept> | joins m.dept = r.dept


## 3. 走一遍结构 —— branches → occurrences → atoms(裸变量)→ joins

In [3]:
for b in structure.branches:
    print(f"branch {b.branch_id}  path={b.path}")
    for o in b.occurrences:
        print(f"  occurrence {o.occurrence_alias!r}  rule={o.rule_id}  role={o.role}")
        for a in o.atoms:
            terms = getattr(a.form, "terms", getattr(a.form, "operands", ())) if a.form else ()
            naked = [f"{t.port_name}:{t.name}" if isinstance(t, FreeVar) else repr(getattr(t, "value", t)) for t in terms]
            print(f"     {a.atom_id}  [{a.kind}]  {a.summary}   terms={naked}")
    for j in b.joins:
        print(f"  join {j.join_id}: {j.left.occurrence_alias}.{j.left.port_name} == {j.right.occurrence_alias}.{j.right.port_name}")
    print()

branch c0  path=(0, 0, 0, 1)
  occurrence 'staffed_dept'  rule=staffed_dept  role=head
     c0:atom:6  [cmp]  $__head__hd eq 'eng'   terms=[]
  occurrence 'm'  rule=manager  role=body
     c0:atom:0  [field_predicate]  person:role($m__m, $m__mr)   terms=['mgr:$m__m', 'None:$m__mr']
     c0:atom:1  [cmp]  $m__mr eq 'manager'   terms=[]
     c0:atom:2  [field_predicate]  person:dept($m__m, $m__md)   terms=['mgr:$m__m', 'dept:$m__md']
  occurrence 'r'  rule=report  role=body
     c0:atom:3  [field_predicate]  person:role($r__r, $r__rr)   terms=['rep:$r__r', 'None:$r__rr']
     c0:atom:4  [cmp]  $r__rr eq 'report'   terms=[]
     c0:atom:5  [field_predicate]  person:dept($r__r, $r__rd)   terms=['rep:$r__r', 'dept:$r__rd']
  join c0:m.dept=r.dept: m.dept == r.dept

branch c1  path=(1,)
  occurrence 'staffed_dept'  rule=staffed_dept  role=head
     c1:atom:3  [cmp]  $__head__hd eq 'eng'   terms=[]
  occurrence 'x'  rule=executive  role=body
     c1:atom:0  [field_predicate]  person:role($x__

## 4. 与 explain 对位

同一棵树:`RuleStructure`(裸变量,无 verdict)↔ `explain` 的 `EvidenceGraph`(执行值 + verdict),按身份键对齐。

In [4]:
def ids(node, *, static):
    if static:
        return {"branch_id":{b.branch_id for b in node.branches},
                "occurrence_alias":{o.occurrence_alias for b in node.branches for o in b.occurrences},
                "atom_id":{a.atom_id for b in node.branches for o in b.occurrences for a in o.atoms},
                "join_id":{j.join_id for b in node.branches for j in b.joins}}
    p = node.paths
    return {"branch_id":{x.tree_id for x in p},
            "occurrence_alias":{r.occurrence_alias for x in p for r in x.rules},
            "atom_id":{a.atom_id for x in p for r in x.rules for a in r.atoms},
            "join_id":{j.join_id for x in p for j in x.joins}}

sk = ids(structure, static=True)
for engine in ("native", "problog", "souffle"):
    ev = fg.eval.explain(expr, head=head, engine=engine).evidence
    same = {k: sk[k] == ids(ev, static=False)[k] for k in sk}
    print(f"{engine:8} node-identical: {same}")

native   node-identical: {'branch_id': True, 'occurrence_alias': True, 'atom_id': True, 'join_id': True}


problog  node-identical: {'branch_id': True, 'occurrence_alias': True, 'atom_id': True, 'join_id': True}


souffle  node-identical: {'branch_id': True, 'occurrence_alias': True, 'atom_id': True, 'join_id': True}


### 4b. 逐节点:display(裸)↔ explanation(执行)

分支 `c0` 里,同一个 `atom_id` 两侧对照——左边变量、右边实际值 + verdict。

In [5]:
ev = fg.eval.explain(expr, head=head, engine="native").evidence
c0_s = next(b for b in structure.branches if b.branch_id == "c0")
c0_e = next(p for p in ev.paths if p.tree_id == "c0")
struct_atoms = {a.atom_id: a for o in c0_s.occurrences for a in o.atoms}
ev_atoms = {a.atom_id: a for r in c0_e.rules for a in r.atoms}
for aid in sorted(struct_atoms):
    s = struct_atoms[aid]; e = ev_atoms.get(aid)
    print(aid)
    print(f"   display : {s.summary}")
    print(f"   explain : {type(e.verdict).__name__ if e else '-':9} {e.repr_text if e else ''}")

c0:atom:0
   display : person:role($m__m, $m__mr)
   explain : Holds     Person alice has role manager
c0:atom:1
   display : $m__mr eq 'manager'
   explain : Holds     manager equals manager
c0:atom:2
   display : person:dept($m__m, $m__md)
   explain : Holds     Person alice has dept eng
c0:atom:3
   display : person:role($r__r, $r__rr)
   explain : Holds     Person bob has role report
c0:atom:4
   display : $r__rr eq 'report'
   explain : Holds     report equals report
c0:atom:5
   display : person:dept($r__r, $r__rd)
   explain : Holds     Person bob has dept eng
c0:atom:6
   display : $__head__hd eq 'eng'
   explain : Holds     eng equals eng


## 5. inspect-floor parity + 引擎中立

`RuleStructure` 是 `fg.rules.inspect` 的**超集**(同样的 `ast` / `render` / `joins` / `templates` …),并且**不依赖事实**——零事实的图给出同样的结构。

In [6]:
insp = fg.rules.inspect(expr)
print("inspect parity  : ast", structure.ast == insp.ast,
      "| render_compact", structure.render_compact() == insp.render_compact(),
      "| templates", structure.templates == insp.templates,
      "| joins", structure.joins == insp.joins)

# engine-neutral: 同样的规则,在一个零事实的 FactGraph 上,结构身份键不变
fg_empty = sdk.SDKStore([Person])
s_empty = fg_empty.rules.structure(expr, head=head)
print("fact-independent: keys identical with zero facts ->", ids(s_empty, static=True) == sk)

inspect parity  : ast True | render_compact True | templates True | joins True
fact-independent: keys identical with zero facts -> True


## 小结

- `fg.rules.structure(rule)` → 引擎中立的 `RuleStructure`:规则的静态结构,**零引擎、零事实依赖**。
- 与 `fg.eval.explain(...)` 的 `EvidenceGraph` **节点恒等对位**(`branch_id`/`occurrence_alias`/`atom_id`/`join_id`),三引擎一致——display(裸)与 explanation(执行)是同一棵树的两个层级。
- 它是 `fg.rules.inspect` 的超集(`ast`/`render`/`joins`/`templates`…),并对 OR 产生 DNF 分支、对 join 暴露 `StructureJoin`。